In [42]:
import pandas as pd
import sqlite3

In [43]:
# old database
conn_old = sqlite3.connect("final_project.db")

# new database
conn_new = sqlite3.connect("final_projectV2.db")

In [44]:
tables = pd.read_sql("""
SELECT name 
FROM sqlite_master 
WHERE type='table';
""", conn_old)

tables

,name
0,area_services
1,hardship
2,areas


In [45]:
for table in tables["name"]:
    
    df = pd.read_sql(f"SELECT * FROM {table};", conn_old)
    
    df.to_sql(
        table,
        conn_new,
        if_exists="replace",
        index=False
    )

print("All tables copied successfully.")

All tables copied successfully.


In [46]:
pd.read_sql("""
SELECT name 
FROM sqlite_master 
WHERE type='table';
""", conn_new)

,name
0,area_final
1,area_services
2,hardship
3,areas


### Query 1: Baseline table showing services and socioeconomic indicators 


In [48]:
# What: Retrieves key indicators for each Chicago community area, including total services,
#      service density, hardship index, poverty rate, and per capita income.
# How: Selects relevant columns from the `area_services` table and orders the results by
#      hardship_index in descending order so that the most socioeconomically disadvantaged
#      neighborhoods appear first.
# Why: This baseline table establishes the core variables used throughout the analysis and
#      provides an initial view of how service access and socioeconomic conditions vary
#      across neighborhoods. It helps identify areas with high hardship that may also
#      experience limited service availability.
q1 = """
SELECT
    area_num,
    area_name,
    total_services,
    services_per_sq_km_broad,
    hardship_index,
    percent_households_below_poverty,
    per_capita_income_
FROM area_final
ORDER BY hardship_index DESC;
"""

pd.read_sql(q1, conn_new).head(15)

,area_num,area_name,total_services,services_per_sq_km_broad,hardship_index,percent_households_below_poverty,per_capita_income_
0,54,Riverdale,0,0.000000,98,56.5,8201
1,37,Fuller Park,1,0.000000,97,51.2,10432
2,30,South Lawndale,7,0.000000,96,30.7,10402
3,68,Englewood,6,0.000000,94,46.6,11888
4,63,Gage Park,1,0.000000,93,23.4,12171
5,26,West Garfield Park,1,0.000000,92,41.7,10934
6,61,New City,7,0.000000,91,29.0,12765
7,67,West Englewood,0,0.000000,89,34.4,11317
8,40,Washington Park,1,0.023599,88,42.1,13785
9,29,North Lawndale,9,0.000000,87,43.1,12034


Interpretation (Q1)

This baseline query displays ND-proxy service availability alongside socioeconomic indicators for each Chicago community area. Sorting by hardship index reveals that many of the city’s highest-hardship neighborhoods—such as Riverdale, Fuller Park, and Englewood—have extremely low or zero measured service density in the dataset. While this does not prove the complete absence of services, it suggests potential access gaps or incomplete OpenStreetMap tagging and highlights these neighborhoods as priority candidates for further analysis.

### Query 2 Summary statistics for service density 


In [49]:
# What: Computes overall descriptive statistics for service density across all
#     Chicago community areas, including the minimum, average, and maximum
#      number of services per square kilometer.
# How: Uses SQL aggregation functions (MIN, AVG, MAX, and COUNT) on the
#      services_per_sq_km_broad variable from the `area_final` table to summarize
#     the distribution of service access.
# Why: These summary statistics provide a benchmark for understanding how
#      unevenly services are distributed across neighborhoods and help establish
#      reference values used later to identify service deserts or underserved areas.
q2 = """
SELECT
    MIN(services_per_sq_km_broad) AS min_density,
    AVG(services_per_sq_km_broad) AS avg_density,
    MAX(services_per_sq_km_broad) AS max_density,
    COUNT(*) AS neighborhoods
FROM area_final;
"""

pd.read_sql(q2, conn_new)

,min_density,avg_density,max_density,neighborhoods
0,0.0,0.01572,0.227854,77


Interpretation (Q2)

This query summarizes the distribution of service density across all 77 Chicago community areas. 
The relatively low average density (0.0157) suggests that service availability is uneven and concentrated in a small number of neighborhoods.

In [50]:
# Q2b: Identify neighborhoods with service density below city average
# What: finds service deserts relative to citywide service density
# How: compares each neighborhood's density to the average density calculated in a subquery
# Why: helps identify areas underserved relative to the city baseline

q2b = """
SELECT area_name, services_per_sq_km_broad
FROM area_final
WHERE services_per_sq_km_broad <
    (SELECT AVG(services_per_sq_km_broad)
     FROM area_final);
"""

pd.read_sql(q2b, conn_new)

,area_name,services_per_sq_km_broad
0,Rogers Park,0.000000
1,West Ridge,0.010160
2,Edison Park,0.000000
3,Norwood Park,0.008199
4,Jefferson Park,0.000000
5,Forest Glen,0.011219
6,North Park,0.014227
7,Albany Park,0.000000
8,Irving Park,0.000000
9,Montclare,0.000000


Interpretation of Query 2b Results

The results indicate that many Chicago neighborhoods have extremely low or zero service density, while only a few areas show measurable access to services. This uneven distribution suggests that service availability is highly concentrated and supports the presence of potential service deserts across the city.

In [51]:
# Q2c: Identify neighborhoods with hardship above the city median
# What: finds neighborhoods facing above-median socioeconomic hardship
# How: compares hardship index to the median value calculated in a subquery
# Why: highlights communities with elevated socioeconomic vulnerability

q2c = """
SELECT area_name, hardship_index
FROM area_final
WHERE hardship_index >
    (SELECT AVG(hardship_index)
     FROM area_final);
"""

pd.read_sql(q2c, conn_new)

,area_name,hardship_index
0,Albany Park,53
1,Montclare,50
2,Belmont Cragin,70
3,Hermosa,71
4,Humboldt park,85
5,Austin,73
6,West Garfield Park,92
7,East Garfield Park,83
8,North Lawndale,87
9,South Lawndale,96


Interpretation of Query 2c Results

Many neighborhoods classified as service deserts also exhibit high hardship index values, often above 80 or 90. This pattern suggests that communities facing greater socioeconomic disadvantage may also experience lower access to services, highlighting potential spatial inequality in service distribution across Chicago.

Several neighborhoods on Chicago’s South and West sides, such as Englewood, Riverdale, and West Garfield Park, appear prominently among high-hardship desert areas, indicating possible geographic clustering of disadvantage and limited service access.

### Query 3: Rank neighborhoods by service density within hardship quartiles
What: Identifies which neighborhoods have the lowest service density relative to others with similar socioeconomic conditions.
--- How: First uses the NTILE window function to divide community areas into four hardship quartiles based on the hardship_index. It then applies the RANK window function within each quartile to order neighborhoods from lowest to highest service density.
---Why: Comparing service access within hardship groups helps determine whether neighborhoods experiencing high socioeconomic hardship also tend to have disproportionately low service availability, revealing potential clusters of service deserts.

In [52]:
q3 = """
WITH base AS (
  SELECT
    af.area_num,
    af.area_name,
    af.hardship_index,
    af.services_per_sq_km_broad,
    NTILE(4) OVER (ORDER BY af.hardship_index) AS hardship_quartile
  FROM area_final AS af
)
SELECT
  area_num,
  area_name,
  hardship_index,
  hardship_quartile,
  services_per_sq_km_broad,
  RANK() OVER (PARTITION BY hardship_quartile ORDER BY services_per_sq_km_broad ASC) AS density_rank_in_quartile
FROM base
ORDER BY hardship_quartile, density_rank_in_quartile;
"""
pd.read_sql(q3, conn_new).head(15)

,area_num,area_name,hardship_index,hardship_quartile,services_per_sq_km_broad,density_rank_in_quartile
0,9,Edison Park,8,1,0.000000,1
1,72,Beverly,12,1,0.000000,1
2,76,O'Hare,24,1,0.000000,1
3,11,Jefferson Park,25,1,0.000000,1
4,10,Norwood Park,21,1,0.008199,5
5,12,Forest Glen,11,1,0.011219,6
6,74,Mount Greenwood,16,1,0.026461,7
7,24,West Town,10,1,0.031357,8
8,8,Near North Side,1,1,0.039126,9
9,77,Edgewater,19,1,0.041280,10


Interpretation of Query 3a Results

Within the lowest-hardship quartile, service density varies widely across neighborhoods. Some areas show zero recorded services while others have relatively higher densities, suggesting that service availability is uneven even among communities with similar socioeconomic conditions.

The fact that several low-hardship neighborhoods still show zero density likely reflects:
limitations of the service dataset collected via Overpass API, or
services concentrated in adjacent neighborhoods rather than within strict boundaries.

In [53]:
# Query 3b: Compare average service density across hardship quartiles
# What: Calculates the average service density for neighborhoods grouped by
#      socioeconomic hardship level.
# How: Uses the NTILE window function to divide community areas into four
#      hardship quartiles based on the hardship_index. The query then groups
#      these quartiles and computes the average services_per_sq_km_broad and
#      the number of neighborhoods in each group.
# Why: This analysis helps determine whether neighborhoods experiencing higher
#      socioeconomic hardship systematically have lower service access, which
#      would indicate potential inequality in the distribution of community
#      services across Chicago.
q3b = """
WITH hardship_groups AS (
SELECT
    area_name,
    hardship_index,
    services_per_sq_km_broad,
    NTILE(4) OVER (ORDER BY hardship_index) AS hardship_quartile
FROM area_final
)

SELECT
    hardship_quartile,
    AVG(services_per_sq_km_broad) AS avg_service_density,
    COUNT(*) AS neighborhoods
FROM hardship_groups
GROUP BY hardship_quartile
ORDER BY hardship_quartile;
"""

pd.read_sql(q3b, conn_new)

,hardship_quartile,avg_service_density,neighborhoods
0,1,0.049003,20
1,2,0.007853,19
2,3,0.001126,19
3,4,0.003145,19


Interpretation of Query 3b Results

Neighborhoods with the lowest hardship levels have the highest average service density, while higher-hardship quartiles show much lower service availability. This pattern suggests that service access may be unevenly distributed and potentially correlated with socioeconomic disadvantage across Chicago neighborhoods.

The large gap between quartile 1 and the other quartiles indicates that service availability may be concentrated in a relatively small number of lower-hardship neighborhoods.

In [54]:
# Query 3c: Identify lowest-service neighborhoods within the highest-hardship quartile
# What: Finds neighborhoods that have the lowest service density among the
#      most socioeconomically disadvantaged areas (hardship quartile 4).
# How: First divides neighborhoods into hardship quartiles using the NTILE
#      window function based on the hardship_index. It then ranks service
#      density within each quartile using the RANK window function and filters
#      the results to only include quartile 4 (the highest hardship group).
# Why: This query focuses the analysis on the most vulnerable neighborhoods to
#      identify which areas may be experiencing both high socioeconomic hardship
#      and limited access to services, highlighting potential service deserts
#      that may require targeted policy attention.
q3c = """
WITH base AS (
  SELECT
    area_num,
    area_name,
    hardship_index,
    services_per_sq_km_broad,
    NTILE(4) OVER (ORDER BY hardship_index) AS hardship_quartile
  FROM area_final
)
SELECT
  area_num,
  area_name,
  hardship_index,
  hardship_quartile,
  services_per_sq_km_broad,
  RANK() OVER (PARTITION BY hardship_quartile ORDER BY services_per_sq_km_broad ASC) AS density_rank_in_quartile
FROM base
WHERE hardship_quartile = 4
ORDER BY density_rank_in_quartile
LIMIT 15;
"""
pd.read_sql(q3c, conn_new)

,area_num,area_name,hardship_index,hardship_quartile,services_per_sq_km_broad,density_rank_in_quartile
0,36,Oakland,78,4,0.0,1
1,47,Burnside,79,4,0.0,1
2,66,Chicago Lawn,80,4,0.0,1
3,34,Armour Square,82,4,0.0,1
4,27,East Garfield Park,83,4,0.0,1
5,23,Humboldt park,85,4,0.0,1
6,29,North Lawndale,87,4,0.0,1
7,67,West Englewood,89,4,0.0,1
8,61,New City,91,4,0.0,1
9,26,West Garfield Park,92,4,0.0,1


Interpretation of Query 3c Results

Many neighborhoods in the highest hardship quartile show zero recorded service density, including areas such as Riverdale, Fuller Park, and Englewood. This pattern suggests that some of Chicago’s most socioeconomically disadvantaged communities may also experience the lowest levels of service access.

Nearly all neighborhoods in quartile 4 have density rank = 1, which occurs because many of them share the same value (0 service density). This indicates that lack of services is widespread within the highest-hardship group rather than limited to a single outlier neighborhood.

### Query 4:  Service density per square kilometer (multi-join + derived metric)


In [55]:
# Q4: Compute service density per square kilometer using multiple tables
# What: Calculates a normalized measure of service access for each community area,
#      defined as the number of services per square kilometer.
# How: Joins three tables—`hardship` (socioeconomic indicators), `area_services`
#      (service counts), and `areas` (geographic size). The query converts
#      `shape_area` from square meters to square kilometers and then divides
#      the service count by the area size to compute service density.
# Why: Community areas vary significantly in physical size, so raw service counts
#      are not directly comparable. Normalizing services by land area allows us to
#      fairly compare access to services across neighborhoods and forms the basis
#      for identifying potential service deserts.
q4 = """
SELECT
  h.area_num,
  h.area_name,
  s.nd_services_broad,
  a.shape_area,
  (a.shape_area / 1000000.0) AS area_sq_km,
  (s.nd_services_broad / (a.shape_area / 1000000.0)) AS services_per_sq_km_broad,
  h.hardship_index
FROM hardship AS h
JOIN area_services AS s
  ON h.area_num = s.area_num
JOIN areas AS a
  ON h.area_num = a.area_num
ORDER BY services_per_sq_km_broad ASC;
"""
pd.read_sql(q4, conn_new).head(10)

,area_num,area_name,nd_services_broad,shape_area,area_sq_km,services_per_sq_km_broad,hardship_index
0,1,Rogers Park,0,5.125990e+07,51.259902,0.0,39
1,11,Jefferson Park,0,6.486816e+07,64.868162,0.0,25
2,14,Albany Park,0,5.354223e+07,53.542231,0.0,53
3,16,Irving Park,0,8.961138e+07,89.611382,0.0,34
4,18,Montclaire,0,2.757639e+07,27.576394,0.0,50
5,20,Hermosa,0,3.260206e+07,32.602059,0.0,71
6,23,Humboldt park,0,1.004809e+08,100.480877,0.0,85
7,25,Austin,0,1.992542e+08,199.254203,0.0,73
8,26,West Garfield Park,0,3.609285e+07,36.092849,0.0,92
9,27,East Garfield Park,0,5.388322e+07,53.883221,0.0,83


Interpretation of Query 4 Results

After normalizing service counts by geographic area, several neighborhoods still show zero service density. This indicates that some communities—regardless of their physical size—have no recorded services in the dataset, suggesting potential service deserts in parts of Chicago.

### Query 5: Compare hardship and service density between desert and non-desert neighborhoods
This tests whether desert neighborhoods actually have higher hardship on average.

In [56]:
# Q5: Compare hardship and service density between desert and non-desert neighborhoods
# What: Calculates the average hardship index and average service density for
#      neighborhoods classified as service deserts versus non-deserts.
# How: Uses a CASE statement to create a desert_status variable that flags
#      neighborhoods as "Desert" if either desert_zero or desert_rank25 equals 1.
#      The query then aggregates the number of neighborhoods, average hardship,
#      and average service density using GROUP BY.
# Why: This analysis tests whether neighborhoods identified as service deserts
#      also experience greater socioeconomic hardship and lower service access,
#      providing evidence for whether service deserts disproportionately affect
#      disadvantaged communities.

q5 = """
SELECT
    CASE
        WHEN desert_zero = 1 OR desert_rank25 = 1 THEN 'Desert'
        ELSE 'Non-desert'
    END AS desert_status,
    COUNT(*) AS neighborhoods,
    AVG(hardship_index) AS avg_hardship,
    AVG(services_per_sq_km_broad) AS avg_service_density
FROM area_final
GROUP BY desert_status;
"""

pd.read_sql(q5, conn_new)

,desert_status,neighborhoods,avg_hardship,avg_service_density
0,Desert,46,61.891304,0.000000
1,Non-desert,31,31.129032,0.039046


Interpretation of Query 5 Results

Service desert neighborhoods show both significantly higher hardship levels and much lower service density compared to non-desert areas. This pattern suggests that limited service access is associated with greater socioeconomic disadvantage across Chicago neighborhoods.

### Query 6: Identify neighborhoods flagged as deserts under both definitions
The project already uses two desert definitions:
desert_zero
desert_rank25

This query finds areas flagged under both, which is a very strong signal.

In [57]:
# Q6: Identify neighborhoods flagged as service deserts under both definitions
# What: Finds neighborhoods that are classified as service deserts under both
#      desert definitions used in the project: desert_zero and desert_rank25.
# How: Filters the `area_final` table to include only areas where both desert
#      indicators equal 1, then orders the results by hardship_index in
#      descending order.
# Why: Areas flagged under both definitions represent the strongest evidence
#      of service deserts, since they have both extremely low service density
#      and fall into the lowest service-access quartile. Highlighting these
#      neighborhoods helps identify communities that may face the greatest
#      combined challenges of socioeconomic hardship and limited service access.
q6 = """
SELECT
    area_name,
    hardship_index,
    services_per_sq_km_broad,
    desert_zero,
    desert_rank25
FROM area_final
WHERE desert_zero = 1
AND desert_rank25 = 1
ORDER BY hardship_index DESC;
"""

pd.read_sql(q6, conn_new)

,area_name,hardship_index,services_per_sq_km_broad,desert_zero,desert_rank25
0,Riverdale,98,0.0,1,1
1,Fuller Park,97,0.0,1,1
2,Gage Park,93,0.0,1,1
3,West Garfield Park,92,0.0,1,1
4,West Englewood,89,0.0,1,1
5,Armour Square,82,0.0,1,1
6,Chicago Lawn,80,0.0,1,1
7,Burnside,79,0.0,1,1
8,Hermosa,71,0.0,1,1
9,Archer Heights,67,0.0,1,1


Interpretation of Query 6 Results

Neighborhoods identified as deserts under both definitions tend to have very high hardship index values and zero recorded service density. This overlap suggests that service deserts are concentrated in some of Chicago’s most socioeconomically disadvantaged communities.

Many of these neighborhoods are located on Chicago’s South and West sides, such as Riverdale, Fuller Park, Englewood, and West Garfield Park, which indicates a possible geographic clustering of both hardship and limited service access

### Query 7: Window function, show overlap of desert definitions and rank deserts by hardship
This query filters to desert areas under either desert definition and ranks them by hardship index. It supports our sensitivity analysis by showing which neighborhoods remain high-need regardless of whether we define deserts as zero-density or bottom-quartile access.

In [58]:
# Q7: Identify desert neighborhoods and rank them by hardship using a window function
# What: Lists neighborhoods classified as service deserts under either desert
#      definition and ranks them by hardship index.
# How: Filters the dataset to include only neighborhoods where desert_zero or
#      desert_rank25 equals 1. It then applies the RANK window function to order
#      these desert areas from highest to lowest hardship index.
# Why: This ranking helps identify which service desert neighborhoods face the
#      greatest socioeconomic challenges, highlighting areas that may require
#      the most urgent policy intervention.
q7 = """
SELECT
  af.area_num,
  af.area_name,
  af.desert_zero,
  af.desert_rank25,
  af.services_per_sq_km_broad,
  af.hardship_index,
  RANK() OVER (ORDER BY af.hardship_index DESC) AS hardship_rank_among_deserts
FROM area_final AS af
WHERE af.desert_zero = 1 OR af.desert_rank25 = 1
ORDER BY hardship_rank_among_deserts;
"""
pd.read_sql(q7, conn_new).head(25)

,area_num,area_name,desert_zero,desert_rank25,services_per_sq_km_broad,hardship_index,hardship_rank_among_deserts
0,54,Riverdale,1,1,0.0,98,1
1,37,Fuller Park,1,1,0.0,97,2
2,30,South Lawndale,1,0,0.0,96,3
3,68,Englewood,1,0,0.0,94,4
4,63,Gage Park,1,1,0.0,93,5
5,26,West Garfield Park,1,1,0.0,92,6
6,61,New City,1,0,0.0,91,7
7,67,West Englewood,1,1,0.0,89,8
8,29,North Lawndale,1,0,0.0,87,9
9,23,Humboldt park,1,0,0.0,85,10


Interpretation of Query 7 Results

This query ranks all “desert” community areas by hardship index and shows whether each area is flagged under Desert Zero (zero ND-proxy service density) and/or Desert Rank25 (bottom quartile of service density with a tie-break). Every area in the list is a zero-density desert, which means these neighborhoods have no measured ND-proxy services per square kilometer in our dataset.

The additional value here is the overlap signal. Areas flagged as desert_rank25 = 1 represent the most consistently underserved group under both definitions. In the top of the ranking, Riverdale and Fuller Park are flagged under both definitions and have the highest hardship scores (98 and 97), placing them at the top of the “highest-need desert” list. Other neighborhoods, like South Lawndale and Englewood, appear as deserts under the zero-density definition but do not fall into the stricter bottom-quartile group, illustrating how desert definitions can change which neighborhoods are prioritized even when the overall pattern remains the same.